### RAG Fusion Pipeline

This notebook demonstrates a complete RAG (Retrieval-Augmented Generation) pipeline built on top of our custom `RAGFusion` class.
The pipeline uses **LLM-generated sub-queries** to improve retrieval quality, then fuses the results using **Reciprocal Rank Fusion (RRF)**.

**Steps covered:**
1. Load the source PDF
2. Split documents into chunks
3. Generate embeddings and store in ChromaDB
4. Create a similarity-search retriever
5. Apply RAG Fusion (sub-query generation + RRF)
6. Augmentation - build context from retrieved documents
7. Generation - produce a grounded answer using an LLM

### Imports & Setup

In [1]:
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

from rag_fusion import RAGFusion

# Load OPENAI_API_KEY from the .env file
load_dotenv()

C:\Users\Uttam\AppData\Local\Temp\ipykernel_5088\136066656.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
e:\Projects\Campusx-Advance-Rag\Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

### Step 1 - Load the PDF

`PyPDFLoader` reads the PDF and returns one `Document` object per page.

In [2]:
loader = PyPDFLoader("notebooklm_rag.pdf")
pages = loader.load()

print(f"Loaded {len(pages)} page(s) from the PDF.")

Loaded 3 page(s) from the PDF.


### Step 2 - Split Documents into Chunks

Large pages are split into smaller, overlapping chunks so that the retriever can surface focused, relevant passages rather than entire pages.

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)

print(f"Split into {len(chunks)} chunk(s).")

Split into 19 chunk(s).


### Step 3 - Embeddings & Vector Store

Each chunk is converted into a dense vector using OpenAI's `text-embedding-3-small` model and stored in a ChromaDB vector store.
This makes semantic similarity search possible at query time.

In [4]:
# embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="notebooklm_rag"
)

print("Vector store created successfully.")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 17954.01it/s]


Vector store created successfully.


### Step 4 - Create the Retriever

We configure a similarity-search retriever with `k=3`, meaning it will return the 3 most relevant chunks for any given query.

In [5]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [6]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000028B5B89FB60>, search_kwargs={'k': 3})

### Step 5 - RAG Fusion

`RAGFusion.from_llm` wires up the LLM to generate multiple sub-queries from the original query.
Each sub-query is sent to the retriever independently, and the results are merged using **Reciprocal Rank Fusion (RRF)** - documents that rank highly across multiple sub-queries bubble to the top.

In [7]:
import os

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

# Build the RAG Fusion pipeline: LLM generates 2 sub-queries, retrieves docs for each, then fuses
rag_fusion = RAGFusion.from_llm(
    llm=llm,
    retriever=retriever,
    num_subqueries=2,
    k=3
)

In [8]:
query = "How does NotebookLM retrieve relevant information from uploaded documents?"

# This generates sub-queries, retrieves docs for each, and returns RRF-ranked results
fused_docs = rag_fusion.invoke(query)

print(f"Retrieved {len(fused_docs)} fused document(s).")
for i, doc in enumerate(fused_docs):
    print(f"\n--- Document {i + 1} ---")
    print(doc.page_content)

Retrieved 3 fused document(s).

--- Document 1 ---
identifies which specific passages from the uploaded documents supported each claim in the answer. These
citations are surfaced to the user as inline references, allowing them to verify the accuracy of the response by
reading the original source material.
NotebookLM also handles cases where the retrieved context is insufficient to answer the question. In such
cases, the system is designed to explicitly acknowledge the limitation and inform the user that the answer

--- Document 2 ---
When a user uploads a document to NotebookLM, the system begins an automatic indexing process. The
document is first parsed to extract its raw text content. For PDFs, this involves optical character recognition
(OCR) if the document contains scanned pages, or direct text extraction for digital PDFs. The extracted text is
then cleaned and normalized to remove formatting artifacts.
Next, the text is split into overlapping chunks using a strategy that preserv

### Step 6 - Augmentation

The retrieved chunks are concatenated into a single context string.
This context will be injected into the generation prompt to ground the LLM's answer.

In [9]:
# Join all retrieved chunks into one context block
context = "\n\n".join([doc.page_content for doc in fused_docs])

print(context)

identifies which specific passages from the uploaded documents supported each claim in the answer. These
citations are surfaced to the user as inline references, allowing them to verify the accuracy of the response by
reading the original source material.
NotebookLM also handles cases where the retrieved context is insufficient to answer the question. In such
cases, the system is designed to explicitly acknowledge the limitation and inform the user that the answer

When a user uploads a document to NotebookLM, the system begins an automatic indexing process. The
document is first parsed to extract its raw text content. For PDFs, this involves optical character recognition
(OCR) if the document contains scanned pages, or direct text extraction for digital PDFs. The extracted text is
then cleaned and normalized to remove formatting artifacts.
Next, the text is split into overlapping chunks using a strategy that preserves semantic coherence. Rather

material provided by the user, making i

### Step 7 - Generation

The context and original query are passed to the LLM via a structured prompt.
The LLM is instructed to answer **only** from the provided context and to say `"I don't know"` if the answer isn't there.

In [10]:
query

'How does NotebookLM retrieve relevant information from uploaded documents?'

In [11]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context provided below to answer the question.
Be clear, concise, and accurate in your response.
If the answer is not present in the context, say "I don't know" - do not make up an answer.

Context:
{context}

Question: {question}

Answer:
""")

# Chain: prompt -> LLM
generation_chain = prompt | llm

response = generation_chain.invoke({"context": context, "question": query})

print(response.content)

NotebookLM finds answers by building an index of the uploaded material and then searching that index for the most relevant passages.  
1. **Parsing** – the uploaded file is first read to extract raw text. For PDFs the system runs OCR on scanned pages or pulls text directly from digital PDFs (source: “When a user uploads a document… the document is first parsed to extract its raw text content. For PDFs, this involves optical character recognition (OCR)…”)  
2. **Cleaning** – the extracted text is cleaned and normalized to remove formatting artifacts (“The extracted text is then cleaned and normalized to remove formatting artifacts.”)  
3. **Chunking** – the cleaned text is split into overlapping chunks that preserve semantic coherence (“Next, the text is split into overlapping chunks using a strategy that preserves semantic coherence.”)  
4. **Indexing & retrieval** – those chunks are indexed so that a query can be matched against them. When a user asks a question, NotebookLM searches t